In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
HERE = %pwd
sys.path.append(os.path.dirname(HERE))

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
    
import numpy as np
import pandas as pd
import copy
import pickle
import time
import collections
from tqdm import tqdm
from collections import defaultdict

In [ ]:
from src import utils
rng = utils.set_seed()

dir_parent = utils.dir_parent
version_exp = utils.version_exp
dir_workspace = f"{dir_parent}/research/TFCSR"

device_emb = utils.device

In [ ]:
def compute(dict_data, emb, dir_emb):
    def _tmp(text_type, dict_text, query=False):
        ner = dict_data["ner"]
        path_emb = f"{dir_emb}/{text_type}_{utils.rename(emb.model_name)}_inst{emb.inst_type}_{ner}.csv"
        emb.load(path_emb, dict_text=dict_text, query=query)

    # candidate items (documents)
    text_type = "candidates"
    _tmp(text_type, dict_data["items"][text_type], query=False)

    # query
    query = True
    
    ## history
    text_type = "history"
    _tmp(text_type, dict_data["items"][text_type], query=query)
    
    # user log
    dd_text_interaction = dict_data["concat"]
    for text_type, dict_text in dd_text_interaction.items():
        _tmp(text_type, dict_text, query=query)
    
    ## user profile
    dd_text_profile = dict_data["profile"]
    if len(dd_text_profile) > 0:
        for text_type, dict_text in dd_text_profile.items():
            _tmp(text_type, dict_text, query=query)


def run(emb, data_name, N_icl=[1,3,5], flag_replace_NER=False):
    from src.data_loader import Loader
    loader = Loader(dir_workspace, version_exp, data_name, N_icl=N_icl, flag_replace_NER=flag_replace_NER)
    dict_data = loader.load_data()
    
    dir_emb = f"{dir_workspace}/embedding_data/{version_exp}/{data_name}"
    os.makedirs(dir_emb, exist_ok=True)

    d_inst = utils.load_inst(emb.model_name, flag_replace_NER=flag_replace_NER)
    for inst_type, inst_text in d_inst.items():
        print(f"{utils.rename(emb.model_name):30} {data_name:30} {inst_type:10} NER{flag_replace_NER}")
        emb.set_instruction_text(inst_type, inst_text)
        compute(dict_data, emb, dir_emb)

In [ ]:
data_names = ["MovieLens", "Job"] + [f"ARD_{a}" for a in [
    "CDs_and_Vinyl", "Movies_and_TV", "Toys_and_Games", "Sports_and_Outdoors"
]]

flag_replace_NER = [False, True][0]
N_icl = [1,3,5]

model_names_emb = [
    "nvidia/llama-embed-nemotron-8b",
    "Qwen/Qwen3-Embedding-0.6B",
    "Qwen/Qwen3-Embedding-8B",
    "BAAI/bge-m3",
    "Alibaba-NLP/gte-modernbert-base",
    "intfloat/multilingual-e5-large",
    "princeton-nlp/sup-simcse-roberta-large",
    "answerdotai/ModernBERT-large",
    "FacebookAI/roberta-large"
]

for model_name_emb in model_names_emb:
    from src.embedding import Embedding
    model_id = f"{dir_parent}/models/embedding_models/{model_name_emb}"
    emb = Embedding(model_id, device_emb)
    for data_name in data_names:
        run(emb, data_name, N_icl=N_icl, flag_replace_NER=flag_replace_NER)